In [2]:
# ================================================
# 02 - DATA CLEANING
# Fixing and standardizing our raw data
# ================================================

import pandas as pd

# Always start by reloading the raw data
# Never modify the original files — we save cleaned
# versions to data/cleaned so raw data stays intact
ratings = pd.read_csv("data/raw/player_ratings.csv")
portal = pd.read_csv("data/raw/portal_players.csv")

print(f"Ratings loaded: {len(ratings)} players")
print(f"Portal loaded: {len(portal)} players")


# -- STEP 1: LOOK AT THE SHIFTED ROWS --
# First let's see exactly how many rows have dates
# sitting in the NIL Value column instead of actual NIL values

# Check which rows have dates in NIL Value column
shifted = portal[portal['NIL Value'].str.contains('/', na=False)]
print(f"Rows with dates in NIL Value column: {len(shifted)}")
print("\nSample of shifted rows:")
print(shifted[['Name', 'Position', 'NIL Value', 'Date Entered']].head(10))



Ratings loaded: 3044 players
Portal loaded: 332 players
Rows with dates in NIL Value column: 250

Sample of shifted rows:
                 Name Position  NIL Value Date Entered
2        Allen Graves       PF  4/10/2026          NaN
4         David Fuchs       PF  4/10/2026          NaN
5     Tijan Saine Jr.       PG  3/30/2026          NaN
6   Robert Miller III       PF  4/10/2026          NaN
8      Cayden Charles       SG  4/11/2026          NaN
11   Mihailo Petrovic       PG  4/11/2026          NaN
13     Sebastian Mack       SG  4/01/2026          NaN
14  Jeremy Dent-Smith       CG  4/10/2026          NaN
16           Jan Vide       CG  4/01/2026          NaN
18   Ja'Quay Randolph        C  3/17/2026          NaN


In [3]:
# -- STEP 2: UNDERSTAND THE SHIFT PATTERN --
# Let's look at a few shifted vs non-shifted rows
# side by side to understand exactly what happened

print("=== NORMAL ROWS (NIL value looks correct) ===")
normal = portal[~portal['NIL Value'].str.contains('/', na=False)]
print(normal[['Name', 'NIL Value', 'Date Entered']].head(5))

print("\n=== SHIFTED ROWS (date in NIL column) ===")
print(shifted[['Name', 'NIL Value', 'Date Entered']].head(5))

print(f"\nNormal rows: {len(normal)}")
print(f"Shifted rows: {len(shifted)}")
print(f"Total: {len(portal)}")


=== NORMAL ROWS (NIL value looks correct) ===
               Name NIL Value Date Entered
0  Milan Momcilovic       $2M    4/12/2026
1  Tounde Yessoufou     $1.4M    4/21/2026
3       Hamad Mousa     $423K    4/08/2026
7   Christian Bliss  Expected          NaN
9  Vyctorius Miller     $272K    4/07/2026

=== SHIFTED ROWS (date in NIL column) ===
                Name  NIL Value Date Entered
2       Allen Graves  4/10/2026          NaN
4        David Fuchs  4/10/2026          NaN
5    Tijan Saine Jr.  3/30/2026          NaN
6  Robert Miller III  4/10/2026          NaN
8     Cayden Charles  4/11/2026          NaN

Normal rows: 82
Shifted rows: 250
Total: 332


In [4]:
# -- STEP 3: FIX THE SHIFTED ROWS --
# If NIL Value contains a '/' it's actually a date
# So we move it to Date Entered and set NIL Value to None

# Make a copy so we never touch the original raw data
portal_clean = portal.copy()

# Find rows where NIL Value is actually a date
is_shifted = portal_clean['NIL Value'].str.contains('/', na=False)

# For those rows:
# Move the date to Date Entered
portal_clean.loc[is_shifted, 'Date Entered'] = portal_clean.loc[is_shifted, 'NIL Value']

# Set NIL Value to None since we don't have it
portal_clean.loc[is_shifted, 'NIL Value'] = None

# Verify the fix
print("=== AFTER FIX - PREVIOUSLY SHIFTED ROWS ===")
print(portal_clean.loc[is_shifted, ['Name', 'NIL Value', 'Date Entered']].head(5))

print(f"\nRows with missing NIL Value: {portal_clean['NIL Value'].isna().sum()}")
print(f"Rows with valid NIL Value: {portal_clean['NIL Value'].notna().sum()}")


=== AFTER FIX - PREVIOUSLY SHIFTED ROWS ===
                Name NIL Value Date Entered
2       Allen Graves      None    4/10/2026
4        David Fuchs      None    4/10/2026
5    Tijan Saine Jr.      None    3/30/2026
6  Robert Miller III      None    4/10/2026
8     Cayden Charles      None    4/11/2026

Rows with missing NIL Value: 251
Rows with valid NIL Value: 81


In [5]:
# -- STEP 4: CONVERT NIL VALUES TO NUMBERS --
# $2M -> 2000000
# $1.4M -> 1400000
# $423K -> 423000
# Expected -> None

def parse_nil_value(val):
    # If empty or Expected, return None
    if pd.isna(val) or val == 'Expected':
        return None
    # Remove the dollar sign
    val = val.replace('$', '').strip()
    # Convert M (millions) to number
    if 'M' in val:
        return float(val.replace('M', '')) * 1_000_000
    # Convert K (thousands) to number
    if 'K' in val:
        return float(val.replace('K', '')) * 1_000
    return None

# Apply the function to every row
portal_clean['NIL_numeric'] = portal_clean['NIL Value'].apply(parse_nil_value)

# Verify
print("=== NIL VALUES CONVERTED ===")
sample = portal_clean[portal_clean['NIL_numeric'].notna()][['Name', 'NIL Value', 'NIL_numeric']].head(10)
print(sample)


=== NIL VALUES CONVERTED ===
               Name NIL Value  NIL_numeric
0  Milan Momcilovic       $2M    2000000.0
1  Tounde Yessoufou     $1.4M    1400000.0
3       Hamad Mousa     $423K     423000.0
9  Vyctorius Miller     $272K     272000.0


In [6]:
# -- STEP 5: CLEAN UP POSITIONS --
# CG (combo guard) -> SG
# Make everything uppercase and stripped of spaces

print("=== BEFORE POSITION CLEAN ===")
print(portal_clean['Position'].value_counts())

# Map CG to SG
position_map = {
    'CG': 'SG',  # Combo guard -> Shooting Guard
}

portal_clean['Position'] = portal_clean['Position'].replace(position_map)
portal_clean['Position'] = portal_clean['Position'].str.strip().str.upper()

print("\n=== AFTER POSITION CLEAN ===")
print(portal_clean['Position'].value_counts())


=== BEFORE POSITION CLEAN ===
SG    94
PF    76
PG    63
SF    48
CG    29
C     21
Name: Position, dtype: int64

=== AFTER POSITION CLEAN ===
SG    123
PF     76
PG     63
SF     48
C      21
Name: Position, dtype: int64


In [7]:
# -- STEP 6: CLEAN UP YEAR COLUMN --

print("=== BEFORE YEAR CLEAN ===")
print(portal_clean['Year'].value_counts())

# Drop the row with "-" as it's bad data
portal_clean = portal_clean[portal_clean['Year'] != '-']

print("\n=== AFTER YEAR CLEAN ===")
print(portal_clean['Year'].value_counts())
print(f"\nTotal players remaining: {len(portal_clean)}")


=== BEFORE YEAR CLEAN ===
SR       96
SO       72
JR       71
FR       45
RS-JR    17
RS-SO    15
RS-FR     8
RS-SR     6
-         1
Name: Year, dtype: int64

=== AFTER YEAR CLEAN ===
SR       96
SO       72
JR       71
FR       45
RS-JR    17
RS-SO    15
RS-FR     8
RS-SR     6
Name: Year, dtype: int64

Total players remaining: 331


In [8]:
# -- STEP 7: MERGE PORTAL PLAYERS WITH RATINGS --
# We connect the two files by matching player names
# 'left' merge means we keep all portal players
# even if they don't have a rating

merged = pd.merge(
    portal_clean,      # left file - portal players
    ratings,           # right file - advanced metrics
    on='Name',         # match rows where Name is the same
    how='left'         # keep all portal players
)

print(f"Portal players: {len(portal_clean)}")
print(f"After merge: {len(merged)}")
print(f"\nPlayers with ratings matched: {merged['BPR'].notna().sum()}")
print(f"Players without ratings: {merged['BPR'].isna().sum()}")

print("\n=== SAMPLE OF MERGED DATA ===")
print(merged[['Name', 'Position', 'Year', 'NIL_numeric', 'BPR', 'DBPR']].head(5))


Portal players: 331
After merge: 333

Players with ratings matched: 172
Players without ratings: 161

=== SAMPLE OF MERGED DATA ===
               Name Position   Year  NIL_numeric   BPR  DBPR
0  Milan Momcilovic       PF     JR    2000000.0  8.50  2.45
1  Tounde Yessoufou       SF     FR    1400000.0  5.08  1.99
2      Allen Graves       PF  RS-FR          NaN  7.64  2.07
3       Hamad Mousa       SG     SO     423000.0  2.08 -0.60
4       David Fuchs       PF     JR          NaN  1.89  0.89


In [9]:
# -- INVESTIGATE THE DUPLICATE ROWS --
# We went from 331 to 333 meaning 2 extra rows appeared
# Find which names matched more than once

duplicate_names = merged[merged.duplicated(subset='Name', keep=False)]
print("=== PLAYERS WITH DUPLICATE MATCHES ===")
print(duplicate_names[['Name', 'Position', 'Team', 'BPR']].to_string())


=== PLAYERS WITH DUPLICATE MATCHES ===
                  Name Position                  Team   BPR
138      Josiah Harris       PF       Cleveland State -0.64
139      Josiah Harris       PF              La Salle -0.69
144          Josh Reed       SG            Penn State  2.08
145          Josh Reed       SG                Drexel -2.99
238        Karim Rtail       PF                   NaN   NaN
239      Tafara Gapare       PF                   NaN   NaN
240   Godslove Nwabude       PF                   NaN   NaN
241    Chendall Weaver       SG                 Texas  3.77
242      Brett Freeman       SG                   NaN   NaN
243         JJ Rembert       PG                   NaN   NaN
244         Max Pikaar       PF     Southern Illinois -1.83
245     Richard Barron       SG                   NaN   NaN
246        Mikkel Tyne       SG              Richmond -0.69
247        Derin Saran       PG             UC Irvine  0.79
248         Jonah Funk       PF                   NaN   NaN
2

In [10]:
# -- INVESTIGATE WHY SO MANY DUPLICATES --

print(f"Portal players before merge: 331")
print(f"Merged rows: {len(merged)}")

# Check if the ratings file itself has duplicate names
ratings_dupes = ratings[ratings.duplicated(subset='Name', keep=False)]
print(f"\nDuplicate names in ratings file: {len(ratings_dupes)}")
print(ratings_dupes[['Name', 'Team', 'BPR']].head(10))


Portal players before merge: 331
Merged rows: 333

Duplicate names in ratings file: 26
                 Name               Team   BPR
223      Isaiah Brown            Florida  5.05
429        Jake Davis           Illinois  3.93
462  Chandler Jackson     Arkansas State  3.80
610        Josh Smith            Liberty  3.09
662    Jaden Daughtry           Richmond  2.93
700   Christian Jones  George Washington  2.80
759        RJ Johnson     Kennesaw State  2.62
936         Josh Reed         Penn State  2.08
941    Caleb Williams         Georgetown  2.06
945  Chandler Jackson          Milwaukee  2.04


In [11]:
# -- STEP 8: FIX DUPLICATE NAMES IN RATINGS --
# Keep only the highest BPR row for each duplicate name
# This way we always get the better player's stats
import pandas as pd
ratings_deduped = ratings.sort_values('BPR', ascending=False)
ratings_deduped = ratings_deduped.drop_duplicates(subset='Name', keep='first')

print(f"Ratings before dedup: {len(ratings)}")
print(f"Ratings after dedup: {len(ratings_deduped)}")

# Now redo the merge with clean ratings file
merged = pd.merge(
    portal_clean,
    ratings_deduped,
    on='Name',
    how='left'
)

print(f"\nPortal players: {len(portal_clean)}")
print(f"After clean merge: {len(merged)}")
print(f"Players with ratings: {merged['BPR'].notna().sum()}")
print(f"Players without ratings: {merged['BPR'].isna().sum()}")


Ratings before dedup: 3044
Ratings after dedup: 3031

Portal players: 331
After clean merge: 331
Players with ratings: 170
Players without ratings: 161


In [12]:
# -- STEP 9: SAVE CLEANED DATA --

merged.to_csv("data/cleaned/portal_players_clean.csv", index=False)

print("Cleaned file saved!")
print(f"\nFinal dataset summary:")
print(f"  Total portal players: {len(merged)}")
print(f"  Players with advanced metrics: {merged['BPR'].notna().sum()}")
print(f"  Players with NIL values: {merged['NIL_numeric'].notna().sum()}")
print(f"\nColumns in clean file:")
for col in merged.columns:
    print(f"  {col}")
    
# -- STEP 10: RENAME CONFUSING COLUMNS --

merged = merged.rename(columns={
    'Rank_x': 'Portal_Rank',
    'Rank_y': 'Ratings_Rank'
})

# Save again with clean column names
merged.to_csv("data/cleaned/portal_players_clean.csv", index=False)

print("Columns renamed and file updated!")
print(merged[['Portal_Rank', 'Name', 'Position', 'Team', 'BPR', 'DBPR', 'NIL_numeric']].head(5))


Cleaned file saved!

Final dataset summary:
  Total portal players: 331
  Players with advanced metrics: 170
  Players with NIL values: 4

Columns in clean file:
  Rank_x
  Name
  Position
  Year
  Height
  Weight
  Hometown
  ON3 Rating
  IND Comparison
  NIL Value
  Date Entered
  NIL_numeric
  Rank_y
  Team
  OBPR
  DBPR
  BPR
  Poss
  Box_OBPR
  Box_DBPR
  Box_BPR
  Adj_Team_Off_Eff
  Adj_Team_Def_Eff
  Adj_Team_Eff_Margin
Columns renamed and file updated!
   Portal_Rank              Name Position           Team   BPR  DBPR  \
0          5.0  Milan Momcilovic       PF     Iowa State  8.50  2.45   
1          6.0  Tounde Yessoufou       SF         Baylor  5.08  1.99   
2          7.0      Allen Graves       PF    Santa Clara  7.64  2.07   
3         86.0       Hamad Mousa       SG       Cal Poly  2.08 -0.60   
4        253.0       David Fuchs       PF  San Francisco  1.89  0.89   

   NIL_numeric  
0    2000000.0  
1    1400000.0  
2          NaN  
3     423000.0  
4          NaN  
